# Import libraries and dataset into environment

In [1]:
import dill
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import math as mt
import shap
from sklearn.metrics import confusion_matrix, roc_curve, auc
from MLstatkit import Bootstrapping
import json
import gc
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd

In [2]:
path = os.getcwd()
sys.path.append(path)

save_file = os.path.join(path, "session.pkl")

with open(save_file, "rb") as f:
    state = dill.load(f)

split_list_valid_smote = state["split_list_valid_smote"]
split_list_valid_smote_final = state["split_list_valid_smote_final"]
split_list_fil_valid_smote_final = state["split_list_fil_valid_smote_final"]
split_list_sim_onset_valid_smote_final = state["split_list_sim_onset_valid_smote_final"]
split_list_vldiag_valid_smote_final = state["split_list_vldiag_valid_smote_final"]
split_list_diag1_valid_smote_final = state["split_list_diag1_valid_smote_final"]
split_list_diag2_valid_smote_final = state["split_list_diag2_valid_smote_final"]

# Define functions

In [3]:
# Define function to train Lasso regression model
def run_single_lasso_split(i, split):
    train_df = split['train'].copy()
    valid_df = split['valid'].copy()
    test_df = split['test'].copy()

    train_df['outcome'] = train_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    valid_df['outcome'] = valid_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)
    test_df['outcome'] = test_df['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = False)

    train_x = train_df.drop(columns = ['outcome'])
    valid_x = valid_df.drop(columns = ['outcome'])
    test_x = test_df.drop(columns = ['outcome'])

    features = list(train_x.columns)
    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    for col in cat_cols:
        train_x[col] = train_x[col].astype('category')
        valid_x[col] = valid_x[col].astype('category')
        test_x[col] = test_x[col].astype('category')

    train_x = pd.get_dummies(train_x, columns = cat_cols, drop_first = False)
    valid_x = pd.get_dummies(valid_x, columns = cat_cols, drop_first = False)
    test_x = pd.get_dummies(test_x, columns = cat_cols, drop_first = False)

    feature_names = train_x.columns.tolist()

    # Force valid/test to have same columns and same order as train
    valid_x = valid_x.reindex(columns = feature_names, fill_value = 0)
    test_x = test_x.reindex(columns = feature_names, fill_value = 0)
    
    mapping = {"Non severe": 0, "Severe": 1}
    
    train_y = train_df['outcome'].map(mapping).astype(int)
    valid_y = valid_df['outcome'].map(mapping).astype(int)
    test_y = test_df['outcome'].map(mapping).astype(int)
    lasso = LogisticRegression(
        penalty = 'l1',  # Perform L1-regularization/Lasso regression
        solver = 'liblinear',
        l1_ratio = 1.0,
        C = 0.1,   # Inverse of regularization strength
        class_weight = {0: 1, 1: 1.25},    # Define wights associated with classes
        max_iter = 1000,
        random_state = 123
    )

    lasso_model = lasso.fit(train_x, train_y)

    # Make predictions on Testing set
    lasso_pred_prob = lasso_model.predict_proba(test_x)[:, 1]

    # Calculate Area Under the ROC curve
    fpr, tpr_curve, _ = roc_curve(test_y, lasso_pred_prob, pos_label = 1)
    auc_val_lasso = auc(fpr, tpr_curve)

    # Store standard structure dictionary
    roc_container = {
        'actual': test_y,
        'probabilities': lasso_pred_prob
    }
    
    # Calculate Area Under the Precision-Recall Curve (AUPRC / PR-AUC)
    auprc_val, prc_ci_lower, prc_ci_upper = Bootstrapping(test_y, lasso_pred_prob, 'pr_auc')

    # Convert predicted probabilities to the labels
    lasso_pred = lasso_model.predict(test_x).astype(int)
    lasso_pred_res = np.where(lasso_pred == 0, "Non severe", "Severe")
    
    # Compute Confusion Matrix (Test)
    tn, fp, fn, tp = confusion_matrix(test_y, lasso_pred, labels = [0, 1]).ravel()
    
    # Format a formal R-styled evaluation matrix lookup dataframe 
    cfm_lasso = pd.DataFrame(
        [[tn, fp], [fn, tp]], 
        index = ["Non severe", "Severe"], 
        columns = ["Non severe", "Severe"]
    )
    cfm_lasso.index.name = 'Prediction'
    cfm_lasso.columns.name = 'Observed'
    
    # Calculate performance metrics
    accuracy_lasso = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0
    sensitivity_lasso = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity_lasso = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv_lasso = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_lasso = tp / (tp + fp) if (tp + fp) > 0 else 0
        
    # Make predictions on Training Set to gather accuracy
    lasso_pred_train_prob = lasso_model.predict_proba(train_x)[:, 1]
    lasso_pred_train = lasso_model.predict(train_x).astype(int)
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(train_y, lasso_pred_train, labels = [0, 1]).ravel()
    accuracy_lasso_train = (tp_tr + tn_tr) / (tn_tr + fp_tr + fn_tr + tp_tr)
    
    masker = shap.maskers.Independent(train_x)
    explainer = shap.LinearExplainer(model = lasso_model, masker = masker)
    shap_values = explainer(train_x).values

    return {
        "model": lasso_model,
        "confusion_matrix": cfm_lasso,
        "accuracy_training": accuracy_lasso_train,
        "accuracy_testing": accuracy_lasso,
        "sensitivity": sensitivity_lasso,
        "specificity": specificity_lasso,
        "precision": precision_lasso,
        "npv": npv_lasso,
        "roc": roc_container,
        "AUC_value": auc_val_lasso,
        "PRC_val": auprc_val,
        "PRC_lower_ci": prc_ci_lower,
        "PRC_upper_ci": prc_ci_upper,
        "SHAP_values": shap_values,
        "prediction": lasso_pred_res,
        "pred_prob": lasso_pred_prob
    }

# Define function to train Lasso regression for 100 times in parallel
def model_func_lasso_tune(data_list):
    
    # Count system resource availability profiles
    num_cores = multiprocessing.cpu_count() - 1
    
    if __name__ == '__main__':
        try:
            result_list = Parallel(n_jobs = num_cores)(
                delayed(run_single_lasso_split)(i, data_list[i]) 
                for i in range(100)
                )
        finally:
            externals.loky.get_reusable_executor().shutdown(wait = True)
            gc.collect()

    return result_list

# Fitting dataset into the model

## Fitting data list without VL information

In [4]:
lasso_fil_list = model_func_lasso_tune(split_list_fil_valid_smote_final)
lasso_fil_met_summary = sum_metric(lasso_fil_list)
lasso_fil_metrics_summary = lasso_fil_met_summary["metric_summary"]
lasso_fil_summary = met_collate_func(lasso_fil_metrics_summary).assign(
    models = "Lasso regression (No VL info & SMOTE)"
)
lasso_fil_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=N

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,36.2211,16.5175,59.5285,Lasso regression (No VL info & SMOTE)
1,AUROC_value,92.1385,83.2469,97.5273,Lasso regression (No VL info & SMOTE)
2,Accuracy,86.8308,80.1888,92.4471,Lasso regression (No VL info & SMOTE)
3,Accuracy_train,87.9117,83.2931,91.9821,Lasso regression (No VL info & SMOTE)
4,NPV,99.4476,98.6274,100.0000,Lasso regression (No VL info & SMOTE)
5,Precision,17.3534,11.1708,25.4234,Lasso regression (No VL info & SMOTE)
6,Sensitivity,84.3000,60.0000,100.0000,Lasso regression (No VL info & SMOTE)
7,Specificity,86.9097,79.7352,93.1620,Lasso regression (No VL info & SMOTE)


## Fitting data list with simulated VL at symptom onset

In [5]:
lasso_vlsymp_sim_list = model_func_lasso_tune(split_list_sim_onset_valid_smote_final)
lasso_vlsymp_sim_met_summary = sum_metric(lasso_vlsymp_sim_list)
lasso_vlsymp_sim_metrics_summary = lasso_vlsymp_sim_met_summary["metric_summary"]
lasso_vlsymp_sim_summary = met_collate_func(lasso_vlsymp_sim_metrics_summary).assign(
    models = "Lasso regression (VL symp simulated & SMOTE)"
)
lasso_vlsymp_sim_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,35.0961,16.4274,61.0831,Lasso regression (VL symp simulated & SMOTE)
1,AUROC_value,91.9813,83.4186,97.7321,Lasso regression (VL symp simulated & SMOTE)
2,Accuracy,86.6586,78.6782,92.4471,Lasso regression (VL symp simulated & SMOTE)
3,Accuracy_train,87.9154,83.3749,92.0093,Lasso regression (VL symp simulated & SMOTE)
4,NPV,99.4360,98.6274,100.0000,Lasso regression (VL symp simulated & SMOTE)
5,Precision,17.1672,11.0167,25.4234,Lasso regression (VL symp simulated & SMOTE)
6,Sensitivity,84.0000,60.0000,100.0000,Lasso regression (VL symp simulated & SMOTE)
7,Specificity,86.7414,78.3411,93.1620,Lasso regression (VL symp simulated & SMOTE)


## Fitting data list with VL at diagnosis

In [6]:
lasso_vldiag_list = model_func_lasso_tune(split_list_vldiag_valid_smote_final)
lasso_vldiag_met_summary = sum_metric(lasso_vldiag_list)
lasso_vldiag_metrics_summary = lasso_vldiag_met_summary["metric_summary"]
lasso_vldiag_summary = met_collate_func(lasso_vldiag_metrics_summary).assign(
    models = "Lasso regression (VL diag & SMOTE)"
)
lasso_vldiag_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/s

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,40.4548,17.9522,64.7000,Lasso regression (VL diag & SMOTE)
1,AUROC_value,92.9492,83.3583,97.8520,Lasso regression (VL diag & SMOTE)
2,Accuracy,87.1420,79.4562,92.6057,Lasso regression (VL diag & SMOTE)
3,Accuracy_train,88.8307,85.0169,92.4714,Lasso regression (VL diag & SMOTE)
4,NPV,99.4584,98.6532,100.0000,Lasso regression (VL diag & SMOTE)
5,Precision,17.8153,11.5994,27.3667,Lasso regression (VL diag & SMOTE)
6,Sensitivity,84.6000,60.0000,100.0000,Lasso regression (VL diag & SMOTE)
7,Specificity,87.2212,78.9642,92.8505,Lasso regression (VL diag & SMOTE)


## Fitting data list with VL at diagnosis & VL at 1-day after diagnosis

In [7]:
lasso_add1_list = model_func_lasso_tune(split_list_diag1_valid_smote_final)
lasso_add1_met_summary = sum_metric(lasso_add1_list)
lasso_add1_metrics_summary = lasso_add1_met_summary["metric_summary"]
lasso_add1_summary = met_collate_func(lasso_add1_metrics_summary).assign(
    models = "Lasso regression (VL diag + 1 & SMOTE)"
)
lasso_add1_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,39.4551,16.8041,65.9939,Lasso regression (VL diag + 1 & SMOTE)
1,AUROC_value,93.2361,84.5413,98.1877,Lasso regression (VL diag + 1 & SMOTE)
2,Accuracy,87.4743,80.0453,92.4471,Lasso regression (VL diag + 1 & SMOTE)
3,Accuracy_train,89.3583,85.6179,92.9660,Lasso regression (VL diag + 1 & SMOTE)
4,NPV,99.4514,98.6124,100.0000,Lasso regression (VL diag + 1 & SMOTE)
5,Precision,18.1258,11.9171,27.2727,Lasso regression (VL diag + 1 & SMOTE)
6,Sensitivity,84.3000,60.0000,100.0000,Lasso regression (VL diag + 1 & SMOTE)
7,Specificity,87.5732,79.7352,92.8349,Lasso regression (VL diag + 1 & SMOTE)


## Fitting data list with VL at diagnosis & VL at 2-days after diagnosis

In [8]:
lasso_add2_list = model_func_lasso_tune(split_list_diag2_valid_smote_final)
lasso_add2_met_summary = sum_metric(lasso_add2_list)
lasso_add2_metrics_summary = lasso_add2_met_summary["metric_summary"]
lasso_add2_summary = met_collate_func(lasso_add2_metrics_summary).assign(
    models = "Lasso regression (VL diag + 2 & SMOTE)"
)
lasso_add2_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=N

bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,40.4784,17.1303,64.0892,Lasso regression (VL diag + 2 & SMOTE)
1,AUROC_value,93.4022,86.2344,98.1238,Lasso regression (VL diag + 2 & SMOTE)
2,Accuracy,87.6284,80.7931,92.7492,Lasso regression (VL diag + 2 & SMOTE)
3,Accuracy_train,89.6085,86.1371,92.9141,Lasso regression (VL diag + 2 & SMOTE)
4,NPV,99.4458,98.6182,100.0000,Lasso regression (VL diag + 2 & SMOTE)
5,Precision,18.2773,12.3661,26.8013,Lasso regression (VL diag + 2 & SMOTE)
6,Sensitivity,84.1000,60.0000,100.0000,Lasso regression (VL diag + 2 & SMOTE)
7,Specificity,87.7383,80.6698,93.3100,Lasso regression (VL diag + 2 & SMOTE)


# Save model trained

In [ ]:
path = os.getcwd()

state = {
    "lasso_fil_list": lasso_fil_list,
    "lasso_vlsymp_sim_list": lasso_vlsymp_sim_list,
    "lasso_vldiag_list": lasso_vldiag_list,
    "lasso_add1_list": lasso_add1_list,
    "lasso_add2_list": lasso_add2_list
}

save_file = os.path.join(path, "lasso_trained.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")

In [ ]:
path = os.getcwd()

state = {
    "lasso_fil_summary": lasso_fil_summary,
    "lasso_vlsymp_sim_summary": lasso_vlsymp_sim_summary,
    "lasso_vldiag_summary": lasso_vldiag_summary,
    "lasso_add1_summary": lasso_add1_summary,
    "lasso_add2_summary": lasso_add2_summary
}

save_file = os.path.join(path, "lasso_metric_summary.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")